# 01 — Load and inspect

**Workflow steps covered:** 1 (frame) and 2 (load + inspect).

We rush through these so we can spend real time on EDA, cleaning, modeling, and evaluation. But don't skip them — every shortcut here turns into a bug or a wrong conclusion later.

## 1. Framing the question

> Given a customer's account attributes, can we predict whether they will churn?

Concretely:

| Decision | Choice | Why |
|---|---|---|
| Task type | Binary classification | The target `Churn` is Yes/No |
| Positive class | `Churn = Yes` | This is the rare, costly event we want to detect |
| Unit of analysis | One row = one customer | Confirm later that `customerID` is unique |
| Business action | Outreach / retention offer | This decides the metric we care about (precision vs recall trade-off) |

Why framing matters: if you skip this and jump to `model.fit`, you'll pick `accuracy` as your metric, get 80% (because most customers don't churn), and ship a useless model. Framing forces you to confront the asymmetric cost.

## 2. Load + inspect

Goal: confirm the data is what we think it is. Check shape, dtypes, missingness, duplicates, unique IDs, target distribution.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

df = pd.read_csv(Path('data') / 'telco_churn.csv')
df.shape

(7043, 21)

In [2]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

**Observe:** look at the dtypes carefully.

- `TotalCharges` will probably come in as `object` (string), not `float64`. That's a real-world gotcha — somewhere in the column there are blank strings. We'll fix it.
- `SeniorCitizen` is `int64` (0/1) while the rest of the binary columns are `object` (Yes/No). Inconsistent encoding.
- Everything else looks reasonable for a first pass.

In [4]:
# Convert TotalCharges. errors='coerce' turns the blanks into NaN so we can see them.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].isna().sum()

np.int64(11)

In [ ]:
#what does the field 'tenure' mean? # It represents the number of months the customer has been with the company.
# We can see that there are 11 customers with a tenure of 0 months, which likely corresponds to the 11 customers with missing TotalCharges. We can drop those rows since they don't have any useful information for our analysis.

#what does the field 'churn' mean? # It represents whether the customer has churned (left the company) or not. It is a binary variable with values 'Yes' or 'No'.

In [5]:
# Inspect those rows. What do they have in common?
df[df['TotalCharges'].isna()][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

,customerID,tenure,MonthlyCharges,TotalCharges,Churn
488,4472-LVYGI,0,52.55,NaN,No
753,3115-CZMZD,0,20.25,NaN,No
936,5709-LVOEQ,0,80.85,NaN,No
1082,4367-NUYAO,0,25.75,NaN,No
1340,1371-DWPAZ,0,56.05,NaN,No
3331,7644-OMVMY,0,19.85,NaN,No
3826,3213-VVOLG,0,25.35,NaN,No
4380,2520-SGTTA,0,20.00,NaN,No
5218,2923-ARZLG,0,19.70,NaN,No
6670,4075-WKNIU,0,73.35,NaN,No


**Observe:** the NaN rows all have `tenure == 0` — brand new customers who haven't been billed yet. This is a **structural** missingness, not random. We'll handle it in the cleaning notebook.

In [6]:
# Other missingness checks
df.isna().sum().sort_values(ascending=False).head(10)

TotalCharges      11
gender             0
SeniorCitizen      0
Partner            0
customerID         0
Dependents         0
tenure             0
MultipleLines      0
PhoneService       0
OnlineSecurity     0
dtype: int64

In [7]:
# Duplicates and unique IDs
print('Duplicate rows:', df.duplicated().sum())
print('customerID unique:', df['customerID'].is_unique)
print('Rows / unique IDs:', len(df), '/', df['customerID'].nunique())

Duplicate rows: 0
customerID unique: True
Rows / unique IDs: 7043 / 7043


In [8]:
# Target distribution — class balance check
df['Churn'].value_counts(normalize=True).round(3)

Churn
No     0.735
Yes    0.265
Name: proportion, dtype: float64

**Observe:** ~27% churn / ~73% no-churn. This is **mild imbalance** — not severe enough to need SMOTE or class weighting tricks on day one, but it does mean **accuracy is a bad metric**. A model that predicts "no" for everyone scores 73% accuracy and is useless. Remember this for the evaluation notebook.

## Summary of what we learned

- 7043 rows × 21 columns, one row per customer, no duplicates.
- `TotalCharges` is dirty (blank strings) but for a meaningful reason (tenure=0 customers).
- `SeniorCitizen` is encoded inconsistently with the other binary columns.
- Target is mildly imbalanced (~27% positive). Pick a metric that respects that.

Next: open `02_eda.ipynb`.